# 05 — Deep Learning on Raw EEG Windows
### 1D CNN, CNN-LSTM and Transformer, under the same recording-independent protocol as notebook 04

**Project:** EEG Attention State Detection (Focused / Unfocused / Drowsy)

The question of this notebook:

> **Can a network that sees the raw `14 channels x 256 samples` window do better than hand-crafted features,
> when both are evaluated on exactly the same held-out recordings?**

"Better" has to be earned under the same rules, so almost every design choice below is driven by what notebooks 03
and 04 actually showed on your data.

### What notebooks 03 and 04 showed (numbers from your own runs)

| Finding | Value |
|---|---|
| Kept windows / recordings | 47,354 windows, 34 recordings |
| Class balance | focused 7,765 (16 %), unfocused 10,706 (23 %), drowsy 28,883 (61 %) |
| Windows per recording | very uneven: from 1 (`record17`) and 30 (`record4`) up to 3,623 (`record32`) |
| Majority-class accuracy | 0.624 (so plain accuracy is nearly useless here) |
| Leakage gap (logreg, balanced accuracy) | random split 0.703 vs. recording-grouped 0.577 |
| Best baseline | Logistic Regression, balanced accuracy 0.605 +/- 0.050 (SVM 0.586, Random Forest 0.478) |
| Hardest class | unfocused (recall 0.48), confused with both neighbours |
| Top logistic-regression features | amplitude features: `O2_rms`, `F3_line_length`, `T7_rms`, `T7_std`, `O1_std` |
| PCA (notebook 03) | the three states overlap heavily, while recordings form visible clusters |

### How this shapes the notebook

1. **Same folds as notebook 04.** Same `StratifiedGroupKFold`, same seed, same row order, verified by an assertion below.
   Differences between baselines and networks are then differences of models, not of splits.
2. **Balanced accuracy is the headline metric**, and the loss is class-weighted, because 61 % of windows are drowsy.
3. **A normalization ablation.** The strongest baseline features are *amplitude* features. Amplitude can be genuine
   physiology (alpha power rises when drowsy) but it can also be slow non-neural change: labels here are a pure
   function of elapsed time, so anything that drifts during a 40-minute session (electrode contact, posture, sweat)
   is perfectly correlated with the label. Training the same network on amplitude-free input (per-window z-score)
   shows how much of the performance depends on amplitude. See section 9 for how to read the outcome.
4. **Per-recording scores exclude tiny recordings.** A recording with 1 or 30 windows cannot give a meaningful score.
5. **Realistic expectations.** With 34 recordings (reportedly about 5 people) and fold-to-fold standard deviations of
   about 0.05 in notebook 04, a network that is 0.02 better than logistic regression is *not* distinguishable from it.
   The notebook prints this comparison explicitly instead of leaving it to interpretation.

### Unchanged caveats

- Evaluation is **recording-independent, not subject-independent** (the recording -> participant mapping is still unknown).
- Labels are **protocol-derived** (about first 10 min focused, next 10 unfocused, remainder drowsy), not verified states.
- No hyper-parameter search, one seed. Treat results as a first honest estimate, not a tuned ceiling.

### Requirements

PyTorch is needed (`pip install torch`, see https://pytorch.org/get-started/locally/ for a CUDA build). A GPU helps but
is not required. **Run once with `QUICK_RUN = True` first** (section 1): it trains one small model on one fold with
two epochs, prints the time per epoch, and lets you estimate the full run before committing to it.

## 0. Imports

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
plt.rcParams["figure.figsize"] = (14, 4)
pd.set_option("display.max_columns", 60)

In [ ]:
# [TORCH-ONLY]
import random

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as e:
    raise ImportError(
        "PyTorch is required for this notebook. Install it from https://pytorch.org/get-started/locally/ "
        "(for example: pip install torch)."
    ) from e

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_VERSION = torch.__version__
print(f"torch {TORCH_VERSION} | device: {DEVICE}")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Configuration

Everything tunable is here. `QUICK_RUN = True` overrides the heavy settings for a smoke test.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONT_DIR = PROCESSED_DIR / "continuous"
BASELINE_DIR = PROCESSED_DIR / "baseline_results"
DL_DIR = PROCESSED_DIR / "deep_learning_results"
DL_DIR.mkdir(parents=True, exist_ok=True)

with open(PROCESSED_DIR / "preprocessing_config.json") as f:
    PP_CONFIG = json.load(f)

LABEL_NAMES = {int(k): v for k, v in PP_CONFIG["label_names"].items()}
LABEL_ORDER = [LABEL_NAMES[i] for i in sorted(LABEL_NAMES)]
ACTIVE_CHANNELS = PP_CONFIG["active_channels"]
FS = PP_CONFIG["fs"]
WIN_SAMPLES = int(round(PP_CONFIG["win_sec"] * FS))
N_CLASSES = len(LABEL_NAMES)

QUICK_RUN = False            # True = smoke test (1 model, 1 fold, 2 epochs, small training subset)

# ---- evaluation protocol: identical to notebook 04
N_SPLITS = 5
CV_SEED = 42

# ---- inputs
INPUT_NORM_MAIN = "robust"   # "robust" = signal exactly as saved by notebook 02 (per-recording robust z-score)
                             # "window_zscore" = each channel of each window re-standardized (amplitude removed)
CLIP_VALUE = 20.0            # clip extreme values to keep gradients sane

# ---- models and training
MODELS_TO_RUN = ["cnn1d", "cnn_lstm", "transformer"]
EPOCHS = 20
PATIENCE = 5                 # early stopping on validation balanced accuracy
BATCH_SIZE = 128
LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
DROPOUT = 0.3
LABEL_SMOOTHING = 0.05
CLASS_WEIGHT_POWER = 1.0     # 1.0 = inverse-frequency weights, 0.5 = softened, 0 = unweighted
EPOCH_FRACTION = 0.5         # each epoch visits a random 50 % of training windows (neighbours overlap by 50 %)
MAX_TRAIN_WINDOWS = None     # optional cap on training windows per fold
INNER_VAL_SPLITS = 6         # ~1/6 of the *training recordings* are held out for early stopping
TRAIN_SEED = 42

# ---- light augmentation (training only)
AUGMENT = True
AUG_NOISE_STD = 0.1
AUG_SCALE_JITTER = 0.1

# ---- extra analyses
RUN_NORM_ABLATION = True
ABLATION_MODEL = "cnn1d"
MIN_WINDOWS_PER_REC_EVAL = 100
FOLDS_TO_RUN = None          # None = all folds

if QUICK_RUN:
    MODELS_TO_RUN = ["cnn1d"]
    EPOCHS = 2
    FOLDS_TO_RUN = [0]
    MAX_TRAIN_WINDOWS = 6000
    RUN_NORM_ABLATION = False

print(f"channels={len(ACTIVE_CHANNELS)}, window={WIN_SAMPLES} samples, classes={LABEL_ORDER}")
print(f"QUICK_RUN={QUICK_RUN} | models={MODELS_TO_RUN} | epochs={EPOCHS} | folds={FOLDS_TO_RUN or 'all'}")

## 2. Load the windows and signals

Only `keep == True` windows from notebook 02 are used. All continuous signals are concatenated into one `float32`
array, and each window is described by its global start index. Batches are built by fancy-indexing this array,
so windows are never duplicated in memory (adjacent windows overlap by 50 %) and no `DataLoader` workers are needed
(which also avoids multiprocessing problems on Windows).

In [ ]:
windows = pd.read_csv(PROCESSED_DIR / "windows.csv")
windows = windows[windows["keep"]].reset_index(drop=True)
windows["label_name"] = windows["label"].map(LABEL_NAMES)
assert (windows["end"] - windows["start"] == WIN_SAMPLES).all(), "Window length differs from the config."

rec_names = list(pd.unique(windows["recording"]))
signals, offsets, rec_len = [], {}, {}
cursor = 0
for rec in rec_names:
    npz = np.load(CONT_DIR / Path(rec).with_suffix(".npz").name)
    sig = npz["signal"].astype(np.float32)
    assert list(npz["channels"]) == ACTIVE_CHANNELS, f"{rec}: channel order differs"
    offsets[rec], rec_len[rec] = cursor, len(sig)
    cursor += len(sig)
    signals.append(sig)
SIGNAL = np.concatenate(signals, axis=0)
del signals

assert (windows["end"] <= windows["recording"].map(rec_len)).all(), "A window extends past its recording."

win_start_global = (windows["recording"].map(offsets).to_numpy() + windows["start"].to_numpy()).astype(np.int64)
y = windows["label"].to_numpy().astype(np.int64)
groups = windows["group"].to_numpy()
rec_of_window = windows["recording"].to_numpy()

print(f"Signal array: {SIGNAL.shape} float32 ({SIGNAL.nbytes / 1e6:.0f} MB)")
print(f"Windows: {len(windows):,} | recordings: {len(rec_names)}")
print(pd.Series(y).map(LABEL_NAMES).value_counts().reindex(LABEL_ORDER).to_string())

In [ ]:
_AR = np.arange(WIN_SAMPLES)


def gather_windows(idx, input_norm, augment=False, rng=None):
    """Return a float32 array (len(idx), n_channels, WIN_SAMPLES) for the given window indices."""
    idx = np.asarray(idx)
    gi = win_start_global[idx][:, None] + _AR[None, :]          # (B, win)
    x = np.ascontiguousarray(SIGNAL[gi].transpose(0, 2, 1))     # (B, C, win)

    if input_norm == "window_zscore":
        mu = x.mean(axis=2, keepdims=True)
        sd = x.std(axis=2, keepdims=True)
        x = (x - mu) / (sd + 1e-6)
    elif input_norm != "robust":
        raise ValueError(f"Unknown input_norm: {input_norm!r}")

    if augment:
        rng = rng if rng is not None else np.random.default_rng()
        if AUG_SCALE_JITTER > 0:
            x = x * (1.0 + rng.uniform(-AUG_SCALE_JITTER, AUG_SCALE_JITTER, size=(len(idx), 1, 1))).astype(np.float32)
        if AUG_NOISE_STD > 0:
            x = x + rng.normal(0.0, AUG_NOISE_STD, size=x.shape).astype(np.float32)

    x = np.clip(x, -CLIP_VALUE, CLIP_VALUE)
    return x.astype(np.float32, copy=False)


xb_test = gather_windows(np.arange(4), "robust")
print("Batch shape (B, channels, samples):", xb_test.shape, "| dtype:", xb_test.dtype)

### 2.1 What the inputs look like

Left: how many kept windows each recording contributes (very uneven, and a few recordings are almost empty, which
is worth a look back at the rejection reasons in notebook 02 section 8.1). Right: one example window per class.

In [ ]:
per_rec = windows.groupby("recording").size().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].bar(range(len(per_rec)), per_rec.values, color=["#d62728" if v < MIN_WINDOWS_PER_REC_EVAL else "#1f77b4" for v in per_rec.values])
axes[0].set_xticks(range(len(per_rec)))
axes[0].set_xticklabels([r.replace("eeg_record", "").replace(".mat", "") for r in per_rec.index], fontsize=7)
axes[0].set_xlabel("recording number")
axes[0].set_ylabel("kept windows")
axes[0].set_title(f"Kept windows per recording (red: fewer than {MIN_WINDOWS_PER_REC_EVAL})")

rng_demo = np.random.default_rng(0)
big_recs = per_rec[per_rec > 1000].index
demo_rec = rng_demo.choice(big_recs) if len(big_recs) else per_rec.index[-1]
ax = axes[1]
t = np.arange(WIN_SAMPLES) / FS
shown = 0
for lab in sorted(LABEL_NAMES):
    cand = windows.index[(windows["recording"] == demo_rec) & (windows["label"] == lab)]
    if len(cand) == 0:
        continue
    i = rng_demo.choice(cand)
    x = gather_windows([i], "robust")[0]
    for c in range(2):
        ax.plot(t, x[c] + shown * 8, lw=0.8)
    ax.text(t[-1] + 0.02, shown * 8, LABEL_NAMES[lab], va="center", fontsize=9)
    shown += 1
ax.set_yticks([])
ax.set_xlabel("time in window (s)")
ax.set_title(f"Example window per class, 2 channels each ({demo_rec})")
plt.tight_layout()
plt.show()

print("Recordings with fewer than", MIN_WINDOWS_PER_REC_EVAL, "kept windows:")
print(per_rec[per_rec < MIN_WINDOWS_PER_REC_EVAL].to_string())

## 3. The folds (identical to notebook 04)

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` on `(y, recording)`. The split depends only on the
labels, the groups and the row order, and all three are the same as in notebook 04. If notebook 04's saved
out-of-fold file is present, we verify that the row order and labels match before trusting this.

In [ ]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_SEED)
folds = list(cv.split(np.zeros(len(y)), y, groups))

rows = []
for i, (tr, te) in enumerate(folds):
    assert not (set(groups[tr]) & set(groups[te])), "Recording leaked across train/test!"
    share = np.bincount(y[te], minlength=N_CLASSES) / len(te)
    rows.append({"fold": i, "train_recordings": len(set(groups[tr])), "test_recordings": len(set(groups[te])),
                 "test_windows": len(te), **{f"test_{LABEL_NAMES[k]}": round(share[k], 3) for k in range(N_CLASSES)}})
display(pd.DataFrame(rows))

oof_path = BASELINE_DIR / "out_of_fold_predictions.csv"
BASELINE_OOF = None
if oof_path.exists():
    BASELINE_OOF = pd.read_csv(oof_path)
    same = (len(BASELINE_OOF) == len(windows)
            and (BASELINE_OOF["recording"].to_numpy() == rec_of_window).all()
            and (BASELINE_OOF["true"].to_numpy() == y).all())
    print("Row order and labels identical to notebook 04's out-of-fold file:", same)
    if not same:
        warnings.warn("Row order differs from notebook 04: folds are NOT guaranteed to match the baselines.")
        BASELINE_OOF = None
else:
    print("No notebook-04 out-of-fold file found; baseline comparison by recording will be skipped.")

## 4. Architectures

All three take `(batch, 14, 256)` and output 3 logits. They are deliberately **similar in size (roughly 50k to 90k
parameters)** so that a comparison reflects the architecture, not the capacity. The count is printed below.

| Model | Idea | What it can capture |
|---|---|---|
| `cnn1d` | 3 conv blocks (Conv, BatchNorm, ELU, MaxPool), global average pooling | local waveform and spectral patterns, anywhere in the window |
| `cnn_lstm` | 2 conv blocks, then a bidirectional LSTM over the 64 remaining time steps, mean-pooled | local patterns plus their temporal order |
| `transformer` | 8-sample patch embedding (32 tokens) + class token + 2 pre-norm encoder layers | long-range interactions between segments of the window |

With only about 5 people's worth of data, expect the *smaller inductive bias* models (Transformer) to be the least
likely to help. That is a prior, not a result; the cross-validation decides.

In [ ]:
# [TORCH-ONLY]
class ConvBlock(nn.Module):
    def __init__(self, c_in, c_out, k, pool=2, p_drop=0.0):
        super().__init__()
        self.conv = nn.Conv1d(c_in, c_out, k, padding=k // 2)     # k is odd: length is preserved
        self.bn = nn.BatchNorm1d(c_out)
        self.pool = nn.MaxPool1d(pool) if pool > 1 else nn.Identity()
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        return self.drop(self.pool(F.elu(self.bn(self.conv(x)))))


class EEGCNN1D(nn.Module):
    def __init__(self, n_channels, n_classes, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(n_channels, 32, 7, pool=2, p_drop=dropout / 2),     # 256 -> 128
            ConvBlock(32, 64, 5, pool=2, p_drop=dropout / 2),             # 128 -> 64
            ConvBlock(64, 128, 5, pool=2, p_drop=dropout / 2),            # 64 -> 32
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(dropout),
                                  nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


class EEGCNNLSTM(nn.Module):
    def __init__(self, n_channels, n_classes, dropout=0.3, hidden=64):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(n_channels, 32, 7, pool=2, p_drop=dropout / 2),     # 256 -> 128
            ConvBlock(32, 64, 5, pool=2, p_drop=dropout / 2),             # 128 -> 64
        )
        self.lstm = nn.LSTM(64, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, x):
        h = self.features(x).transpose(1, 2)        # (B, 64 time steps, 64 features)
        out, _ = self.lstm(h)                       # (B, 64, 2*hidden)
        return self.fc(self.drop(out.mean(dim=1)))


class EEGTransformer(nn.Module):
    def __init__(self, n_channels, n_classes, win_samples, patch=8, d_model=64, n_heads=4,
                 n_layers=2, ff=128, dropout=0.2):
        super().__init__()
        assert win_samples % patch == 0, "window length must be divisible by the patch size"
        n_tokens = win_samples // patch
        self.patch_embed = nn.Conv1d(n_channels, d_model, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, n_tokens + 1, d_model))
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=ff, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        tokens = self.patch_embed(x).transpose(1, 2)                  # (B, n_tokens, d_model)
        cls = self.cls.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos
        tokens = self.encoder(tokens)
        return self.head(self.norm(tokens[:, 0]))


def build_model(name):
    n_ch = len(ACTIVE_CHANNELS)
    if name == "cnn1d":
        return EEGCNN1D(n_ch, N_CLASSES, DROPOUT)
    if name == "cnn_lstm":
        return EEGCNNLSTM(n_ch, N_CLASSES, DROPOUT)
    if name == "transformer":
        return EEGTransformer(n_ch, N_CLASSES, WIN_SAMPLES)
    raise ValueError(f"Unknown model: {name}")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# [TORCH-ONLY]
# Shape check on a real batch, and parameter counts.
xb = torch.from_numpy(gather_windows(np.arange(8), INPUT_NORM_MAIN))
summary_rows = []
for name in ["cnn1d", "cnn_lstm", "transformer"]:
    m = build_model(name).eval()
    with torch.no_grad():
        out = m(xb)
    assert out.shape == (8, N_CLASSES), f"{name}: unexpected output shape {tuple(out.shape)}"
    summary_rows.append({"model": name, "input": tuple(xb.shape), "output": tuple(out.shape), "parameters": count_params(m)})
display(pd.DataFrame(summary_rows))

## 5. Training procedure (one fold)

For each outer fold, the held-out test recordings are **never touched** during training or model selection:

1. The outer training recordings are split again **by recording** into an inner train part and an inner validation
   part (about 1/6 of the recordings). The validation part is used only for early stopping.
2. Loss: cross-entropy with inverse-frequency class weights computed on the inner training part, plus mild label
   smoothing.
3. AdamW, cosine learning-rate schedule, gradient clipping, light augmentation (amplitude jitter and Gaussian noise).
4. Early stopping on **validation balanced accuracy** (patience `PATIENCE`); the best weights are restored.
5. Only then is the outer test fold predicted.

Because the validation set is a handful of recordings, the early-stopping point is itself noisy. That is one more
reason not to over-read small differences between models.

In [ ]:
# [TORCH-ONLY]
def eval_split(model, idx, input_norm, criterion, batch_size=512):
    """Return (mean loss, softmax probabilities) for the windows in idx."""
    model.eval()
    probas, total_loss = [], 0.0
    with torch.no_grad():
        for i in range(0, len(idx), batch_size):
            b = idx[i:i + batch_size]
            xb = torch.from_numpy(gather_windows(b, input_norm)).to(DEVICE)
            yb = torch.from_numpy(y[b]).to(DEVICE)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(b)
            probas.append(torch.softmax(logits, dim=1).cpu().numpy())
    if not probas:
        return float("nan"), np.zeros((0, N_CLASSES), dtype=np.float32)
    return total_loss / len(idx), np.concatenate(probas)


def train_one_fold(model_name, input_norm, train_idx, test_idx, seed=TRAIN_SEED, verbose=True):
    set_seed(seed)
    rng = np.random.default_rng(seed)

    # --- inner validation split, by recording
    n_groups_tr = len(np.unique(groups[train_idx]))
    n_inner = max(2, min(INNER_VAL_SPLITS, n_groups_tr))
    inner = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=seed)
    a, b = next(inner.split(np.zeros(len(train_idx)), y[train_idx], groups[train_idx]))
    tr_idx, val_idx = train_idx[a], train_idx[b]
    assert not (set(groups[tr_idx]) & set(groups[val_idx])), "validation recordings overlap with training"
    assert not (set(groups[train_idx]) & set(groups[test_idx])), "test recordings overlap with training"
    if MAX_TRAIN_WINDOWS and len(tr_idx) > MAX_TRAIN_WINDOWS:
        tr_idx = rng.choice(tr_idx, MAX_TRAIN_WINDOWS, replace=False)

    # --- class weights from the inner training part only
    counts = np.bincount(y[tr_idx], minlength=N_CLASSES).astype(float)
    w = (counts.sum() / (N_CLASSES * np.maximum(counts, 1.0))) ** CLASS_WEIGHT_POWER
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE),
                                    label_smoothing=LABEL_SMOOTHING)

    model = build_model(model_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    n_per_epoch = max(BATCH_SIZE, int(len(tr_idx) * EPOCH_FRACTION))

    best = {"score": -np.inf, "state": None, "epoch": 0}
    history, bad, t0 = [], 0, time.time()

    for epoch in range(1, EPOCHS + 1):
        t_ep = time.time()
        model.train()
        order = rng.permutation(tr_idx)[:n_per_epoch]
        running, seen = 0.0, 0
        for i in range(0, len(order), BATCH_SIZE):
            bidx = order[i:i + BATCH_SIZE]
            if len(bidx) < 2:                     # BatchNorm needs more than one sample
                continue
            xb = torch.from_numpy(gather_windows(bidx, input_norm, augment=AUGMENT, rng=rng)).to(DEVICE)
            yb = torch.from_numpy(y[bidx]).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            running += loss.item() * len(bidx)
            seen += len(bidx)
        scheduler.step()

        val_loss, val_proba = eval_split(model, val_idx, input_norm, criterion)
        val_bal = balanced_accuracy_score(y[val_idx], val_proba.argmax(1))
        history.append({"epoch": epoch, "train_loss": running / max(seen, 1), "val_loss": val_loss,
                        "val_bal_acc": float(val_bal)})
        if verbose:
            print(f"  epoch {epoch:2d} | train loss {running / max(seen, 1):.3f} | val loss {val_loss:.3f} "
                  f"| val bal.acc {val_bal:.3f} | {time.time() - t_ep:.1f}s")

        if val_bal > best["score"] or best["state"] is None:
            best = {"score": float(val_bal), "epoch": epoch,
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                if verbose:
                    print(f"  early stop (best epoch {best['epoch']})")
                break

    model.load_state_dict(best["state"])
    _, proba = eval_split(model, test_idx, input_norm, criterion)
    return {"proba": proba, "history": history, "best_epoch": best["epoch"],
            "train_time_s": time.time() - t0, "n_train": len(tr_idx), "n_val": len(val_idx)}

## 6. Cross-validated runs

`run_cv` trains one model on each outer fold and stores the **out-of-fold probabilities**: every window's prediction
comes from a model that never saw that window's recording (nor the recordings used for early stopping on it).
Out-of-fold probabilities are saved after each model, so a long run is not lost if it is interrupted.

In [ ]:
def run_cv(model_name, input_norm):
    n = len(y)
    oof_proba = np.full((n, N_CLASSES), np.nan, dtype=np.float32)
    fold_rows, histories = [], []
    fold_ids = list(range(len(folds))) if FOLDS_TO_RUN is None else list(FOLDS_TO_RUN)

    for f in fold_ids:
        tr, te = folds[f]
        print(f"[{model_name} | {input_norm}] fold {f + 1}/{len(folds)}: "
              f"{len(tr):,} train windows, {len(te):,} test windows ({len(set(groups[te]))} test recordings)")
        res = train_one_fold(model_name, input_norm, tr, te)
        oof_proba[te] = res["proba"]
        pred = res["proba"].argmax(1)
        fold_rows.append({
            "model": model_name, "input_norm": input_norm, "fold": f,
            "n_test_windows": len(te), "n_test_recordings": len(set(groups[te])),
            "best_epoch": res["best_epoch"], "train_time_s": round(res["train_time_s"], 1),
            "accuracy": accuracy_score(y[te], pred),
            "balanced_accuracy": balanced_accuracy_score(y[te], pred),
            "f1_macro": f1_score(y[te], pred, average="macro"),
        })
        histories.append({"fold": f, "history": res["history"]})
        last = fold_rows[-1]
        print(f"  -> test balanced accuracy {last['balanced_accuracy']:.3f} | accuracy {last['accuracy']:.3f} "
              f"| macro-F1 {last['f1_macro']:.3f} | {last['train_time_s']:.0f}s")

    return {"model": model_name, "input_norm": input_norm, "oof_proba": oof_proba,
            "fold_metrics": pd.DataFrame(fold_rows), "histories": histories}


def save_oof(result):
    np.save(DL_DIR / f"oof_proba__{result['model']}__{result['input_norm']}.npy", result["oof_proba"])


def summarize(results):
    rows = {}
    for (name, norm), r in results.items():
        fm = r["fold_metrics"]
        rows[f"{name} [{norm}]"] = {
            "accuracy_mean": fm["accuracy"].mean(), "accuracy_std": fm["accuracy"].std(ddof=0),
            "balanced_accuracy_mean": fm["balanced_accuracy"].mean(), "balanced_accuracy_std": fm["balanced_accuracy"].std(ddof=0),
            "f1_macro_mean": fm["f1_macro"].mean(), "f1_macro_std": fm["f1_macro"].std(ddof=0),
            "n_folds": len(fm),
        }
    return pd.DataFrame(rows).T

In [ ]:
RESULTS = {}
for name in MODELS_TO_RUN:
    print("=" * 78)
    print(f"MODEL: {name}   INPUT: {INPUT_NORM_MAIN}")
    RESULTS[(name, INPUT_NORM_MAIN)] = run_cv(name, INPUT_NORM_MAIN)
    save_oof(RESULTS[(name, INPUT_NORM_MAIN)])

MAIN_KEYS = [(name, INPUT_NORM_MAIN) for name in MODELS_TO_RUN]
dl_summary = summarize({k: RESULTS[k] for k in MAIN_KEYS})
display(dl_summary.round(3))

## 7. Comparison with the baselines of notebook 04

Same folds, same metrics, standard deviations computed over the same five outer folds. Dashed line: chance for
3 classes. The majority-class baseline reaches balanced accuracy 1/3 by construction.

In [ ]:
baseline_path = BASELINE_DIR / "model_comparison.csv"
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path, index_col=0)
    baseline_df["family"] = "baseline (handcrafted features)"
else:
    baseline_df = pd.DataFrame()
    print("notebook-04 results not found: showing the deep models only.")

dl_df = dl_summary.copy()
dl_df["family"] = "deep (raw EEG)"
cols = ["accuracy_mean", "accuracy_std", "balanced_accuracy_mean", "balanced_accuracy_std",
        "f1_macro_mean", "f1_macro_std", "family"]
comparison = pd.concat([baseline_df[cols] if len(baseline_df) else None, dl_df[cols]]).sort_values(
    "balanced_accuracy_mean", ascending=False)
display(comparison.round(3))

fig, ax = plt.subplots(figsize=(10, 0.55 * len(comparison) + 1.5))
order = comparison.sort_values("balanced_accuracy_mean", ascending=True)
colors = ["#ff7f0e" if f.startswith("deep") else "#1f77b4" for f in order["family"]]
ax.barh(order.index, order["balanced_accuracy_mean"], xerr=order["balanced_accuracy_std"], capsize=3, color=colors)
ax.axvline(1 / N_CLASSES, ls="--", c="gray", label="chance (3 classes)")
ax.set_xlabel("Balanced accuracy (recording-grouped CV, mean +/- std over folds)")
ax.set_title("Handcrafted features (blue) vs. raw-EEG networks (orange)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_dl_name = dl_summary["balanced_accuracy_mean"].idxmax()
dl_best = dl_summary.loc[best_dl_name]
print(f"Best network: {best_dl_name}: balanced accuracy {dl_best['balanced_accuracy_mean']:.3f} +/- {dl_best['balanced_accuracy_std']:.3f}")

if len(baseline_df):
    real_baselines = baseline_df.drop(index=[i for i in baseline_df.index if "majority" in i], errors="ignore")
    best_base_name = real_baselines["balanced_accuracy_mean"].idxmax()
    base_best = real_baselines.loc[best_base_name]
    diff = dl_best["balanced_accuracy_mean"] - base_best["balanced_accuracy_mean"]
    spread = max(dl_best["balanced_accuracy_std"], base_best["balanced_accuracy_std"])
    print(f"Best baseline: {best_base_name}: balanced accuracy {base_best['balanced_accuracy_mean']:.3f} +/- {base_best['balanced_accuracy_std']:.3f}")
    print(f"Difference (network - baseline): {diff:+.3f}   | larger fold-to-fold std: {spread:.3f}")
    if abs(diff) < spread:
        print("-> The difference is smaller than the fold-to-fold variation: with 5 folds and one seed, the two are NOT distinguishable.")
    elif diff > 0:
        print("-> The network is ahead by more than the fold-to-fold variation. Still check the per-recording view (section 8) before concluding.")
    else:
        print("-> The baseline is ahead by more than the fold-to-fold variation: the networks do not (yet) add value over handcrafted features here.")

## 8. Where do the networks succeed and fail?

### 8.1 Pooled out-of-fold confusion matrices

Row-normalized, so each row shows how a true class is distributed over predictions. Compare with the baseline matrix
of notebook 04 (focused 0.62, unfocused 0.48, drowsy 0.69 on the diagonal). Of particular interest: does any model fix
the **unfocused** class, or does it still get absorbed by its neighbours?

In [ ]:
fig, axes = plt.subplots(1, len(MAIN_KEYS), figsize=(5.2 * len(MAIN_KEYS), 5), squeeze=False)
for ax, key in zip(axes[0], MAIN_KEYS):
    proba = RESULTS[key]["oof_proba"]
    mask = ~np.isnan(proba[:, 0])
    pred = proba[mask].argmax(1)
    cm = confusion_matrix(y[mask], pred, labels=list(range(N_CLASSES)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{key[0]} (out-of-fold)")
plt.tight_layout()
plt.show()

for key in MAIN_KEYS:
    proba = RESULTS[key]["oof_proba"]
    mask = ~np.isnan(proba[:, 0])
    print(f"\n{key[0]} [{key[1]}]")
    print(classification_report(y[mask], proba[mask].argmax(1), labels=list(range(N_CLASSES)),
                                target_names=LABEL_ORDER, zero_division=0))

### 8.2 Training curves

Validation balanced accuracy per epoch (inner validation recordings, one line per outer fold). Curves that peak early
and then fall mean the model starts memorizing the training recordings; very noisy curves reflect the small number
of validation recordings.

In [ ]:
fig, axes = plt.subplots(1, len(MAIN_KEYS), figsize=(5.2 * len(MAIN_KEYS), 3.8), squeeze=False, sharey=True)
for ax, key in zip(axes[0], MAIN_KEYS):
    for h in RESULTS[key]["histories"]:
        ep = [r["epoch"] for r in h["history"]]
        ax.plot(ep, [r["val_bal_acc"] for r in h["history"]], marker="o", ms=3, label=f"fold {h['fold']}")
    ax.axhline(1 / N_CLASSES, ls="--", c="gray")
    ax.set_title(key[0])
    ax.set_xlabel("epoch")
axes[0, 0].set_ylabel("validation balanced accuracy")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

### 8.3 Per recording: best network vs. the notebook-04 baseline

Each point is one recording scored on its own windows (out-of-fold, so it was held out when predicted).
Points **above** the diagonal are recordings where the network does better than the logistic-regression baseline.
Recordings with fewer than `MIN_WINDOWS_PER_REC_EVAL` windows, or with a single class present, are excluded because
their balanced accuracy is not meaningful (notebook 04's leave-one-recording-out table contained such cases, for
example a recording with 1 window).

In [ ]:
best_key = max(MAIN_KEYS, key=lambda k: RESULTS[k]["fold_metrics"]["balanced_accuracy"].mean())
proba = RESULTS[best_key]["oof_proba"]
have = ~np.isnan(proba[:, 0])
dl_pred = proba.argmax(1)

rows, excluded = [], []
for rec, g in windows.groupby("recording"):
    ii = g.index.to_numpy()
    ii = ii[have[ii]]
    if len(ii) < MIN_WINDOWS_PER_REC_EVAL or len(np.unique(y[ii])) < 2:
        excluded.append((rec, len(ii)))
        continue
    row = {"recording": rec, "n_windows": len(ii),
           "dl_balanced_accuracy": balanced_accuracy_score(y[ii], dl_pred[ii])}
    if BASELINE_OOF is not None:
        row["baseline_balanced_accuracy"] = balanced_accuracy_score(y[ii], BASELINE_OOF["pred"].to_numpy()[ii])
    rows.append(row)

rec_cols = ["recording", "n_windows", "dl_balanced_accuracy"] + (["baseline_balanced_accuracy"] if BASELINE_OOF is not None else [])
per_rec_df = pd.DataFrame(rows, columns=rec_cols).sort_values("dl_balanced_accuracy")
display(per_rec_df.round(3))
print(f"Excluded from this view ({len(excluded)} recordings): {[(r.replace('eeg_record', '').replace('.mat', ''), n) for r, n in excluded]}")

if "baseline_balanced_accuracy" in per_rec_df and len(per_rec_df):
    fig, ax = plt.subplots(figsize=(5.8, 5.5))
    ax.scatter(per_rec_df["baseline_balanced_accuracy"], per_rec_df["dl_balanced_accuracy"],
               s=np.clip(per_rec_df["n_windows"] / 15, 15, 200), alpha=0.7)
    lim = [0, 1]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("baseline balanced accuracy (notebook 04 best model)")
    ax.set_ylabel(f"{best_key[0]} balanced accuracy")
    ax.set_title("Per-recording comparison (point size ~ number of windows)")
    plt.tight_layout()
    plt.show()
    better = (per_rec_df["dl_balanced_accuracy"] > per_rec_df["baseline_balanced_accuracy"]).sum()
    print(f"Network better than baseline on {better} of {len(per_rec_df)} evaluated recordings.")

## 9. Ablation: does the performance depend on amplitude?

We train the same architecture again with **per-window, per-channel z-scoring** as input. This removes the absolute
amplitude and any amplitude change over the session, leaving only the *shape* of the signal (spectral content, waveform).

How to read the outcome (this is guidance, not a verdict):

- **Little change:** the model is mostly using shape information (spectral/temporal structure), which is the physiological
  story we hope for.
- **Clear drop:** amplitude was an important cue. That is *consistent with* real effects (alpha power rises and
  changes the amplitude when drowsy) **and equally consistent with** slow non-neural drift that tracks elapsed time,
  which is the label. This experiment cannot tell the two apart; it only tells you how much depends on amplitude.
  If this happens, say so in the report instead of attributing all performance to attention.
- **Clear gain:** amplitude differences between recordings were hurting generalization, and the per-recording robust
  scaling from notebook 02 was not removing enough of them.

In [ ]:
if RUN_NORM_ABLATION:
    other = "window_zscore" if INPUT_NORM_MAIN == "robust" else "robust"
    key = (ABLATION_MODEL, other)
    if key not in RESULTS:
        print("=" * 78)
        print(f"ABLATION: {ABLATION_MODEL} with input normalization '{other}'")
        RESULTS[key] = run_cv(ABLATION_MODEL, other)
        save_oof(RESULTS[key])

    ab = summarize({k: v for k, v in RESULTS.items() if k[0] == ABLATION_MODEL})
    display(ab.round(3))

    main = RESULTS[(ABLATION_MODEL, INPUT_NORM_MAIN)]["fold_metrics"]["balanced_accuracy"]
    alt = RESULTS[key]["fold_metrics"]["balanced_accuracy"]
    delta = alt.mean() - main.mean()
    spread = max(main.std(ddof=0), alt.std(ddof=0))
    print(f"\nBalanced accuracy with '{other}' minus with '{INPUT_NORM_MAIN}': {delta:+.3f} (fold-to-fold std about {spread:.3f})")
    if abs(delta) < spread:
        print("-> Within fold-to-fold variation: no clear dependence on amplitude in this experiment.")
    elif delta < 0:
        print("-> Dropped: the model relied on amplitude information (see the interpretation guide above).")
    else:
        print("-> Improved: amplitude differences between recordings were hurting generalization.")
else:
    print("RUN_NORM_ABLATION is False: skipped.")

## 10. Save results

In [ ]:
all_summary = summarize(RESULTS)
all_summary.to_csv(DL_DIR / "dl_model_comparison.csv")
pd.concat([r["fold_metrics"] for r in RESULTS.values()]).to_csv(DL_DIR / "dl_fold_metrics.csv", index=False)

oof_table = pd.DataFrame({"recording": rec_of_window, "true": y})
for (name, norm), r in RESULTS.items():
    oof_table[f"pred__{name}__{norm}"] = np.where(np.isnan(r["oof_proba"][:, 0]), -1, r["oof_proba"].argmax(1))
oof_table.to_csv(DL_DIR / "dl_oof_predictions.csv", index=False)

with open(DL_DIR / "dl_histories.json", "w") as f:
    json.dump({f"{n}|{m}": r["histories"] for (n, m), r in RESULTS.items()}, f)

run_config = {
    "torch": TORCH_VERSION, "quick_run": QUICK_RUN, "models": MODELS_TO_RUN, "input_norm_main": INPUT_NORM_MAIN,
    "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT, "label_smoothing": LABEL_SMOOTHING, "class_weight_power": CLASS_WEIGHT_POWER,
    "epoch_fraction": EPOCH_FRACTION, "augment": AUGMENT, "n_splits": N_SPLITS, "cv_seed": CV_SEED,
    "train_seed": TRAIN_SEED, "folds_run": FOLDS_TO_RUN,
}
with open(DL_DIR / "dl_run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

print("Saved to", DL_DIR)
for p in sorted(DL_DIR.glob("*")):
    print(" ", p.name)

## 11. Summary and what's next

**What to extract from the real run** (the numbers depend on your 34 recordings, not on anything written above):

- [ ] Best network vs. best baseline, and whether the gap exceeds the fold-to-fold std (section 7 prints this)
- [ ] Whether any network improves the **unfocused** class (section 8.1)
- [ ] Whether networks beat the baseline on most recordings or only on a few (section 8.3)
- [ ] The amplitude ablation outcome (section 9), reported with its caveat
- [ ] Whether the training curves show overfitting to the training recordings (section 8.2)

**Files written** to `data/processed/deep_learning_results/`

| File | Content |
|---|---|
| `dl_model_comparison.csv` | mean/std accuracy, balanced accuracy, macro-F1 per model and input normalization |
| `dl_fold_metrics.csv` | one row per model, normalization and fold |
| `dl_oof_predictions.csv` | true label and out-of-fold prediction of every window, per model |
| `oof_proba__*.npy` | out-of-fold class probabilities (for calibration, ROC or ensembling later) |
| `dl_histories.json`, `dl_run_config.json` | training curves and exact settings |

**Limitations to state in any write-up**

- Recording-independent, **not** subject-independent evaluation (the recording -> participant mapping is unresolved).
- Labels are protocol-derived and perfectly confounded with elapsed time in the session.
- One seed, no hyper-parameter search; the early-stopping point is chosen on a few validation recordings.
- Several recordings contribute very few windows after cleaning, so a handful of recordings carry most of the data.

**Suggestions that follow from these limits (not run here)**

- Re-run with 3 to 5 seeds and report mean and spread, since differences of a few points are within noise.
- Investigate why some recordings lose almost all windows in notebook 02 before drawing conclusions about them.
- Once the subject mapping exists, switch `groups` to subjects and repeat everything.

**Next:** `06_results_visualization.ipynb`: consolidated figures for the report (baseline vs. network comparison,
per-recording results, confusion matrices) and interpretability (channel-occlusion importance, input-gradient
saliency by frequency band) using the out-of-fold predictions saved here.